In [ ]:
PER_API_KEY = ''

In [130]:
from typing import TypedDict, Optional, List, Dict

class FinancialAdvisorState(TypedDict):

    # User input
    data_path: str
    user_id: str
    user_query: str
    
    time_period: Optional[str]

    # Financial analysis
    financial_summary: Optional[Dict]
    risk_flags: Optional[List[str]]
    simulation_results: Optional[Dict]

    # Agent control
    next_step: Optional[str]
    iteration_count: int
    confidence_score: Optional[float]

    # Final output
    personalized_advice: Optional[str]
    
    # Optional advanced features
    clarification_needed: Optional[bool]
    clarification_question: Optional[str]

In [131]:
import pandas as pd

def analyze_finances(state):

    df = pd.read_csv(state["data_path"])

    # Filter for the specific user
    df = df[df["user_id"] == state["user_id"]]

    # Total income (amount > 0)
    income = df[df["amount"] > 0]["amount"].sum()

    # Total expenses (amount < 0)
    expenses = abs(df[df["amount"] < 0]["amount"].sum())

    savings = income - expenses
    savings_rate = savings / income if income > 0 else 0

    # Category-wise expense breakdown
    category_spend_df = (
        df[df["amount"] < 0]
        .groupby("category")["amount"]
        .sum()
        .abs()
        .sort_values(ascending=False)
    )

    category_spend = category_spend_df.to_dict()

    summary = {
        "total_income": income,
        "total_expenses": expenses,
        "net_savings": savings,
        "savings_rate": round(savings_rate, 2),
        "top_expense_categories": category_spend
    }

    state["financial_summary"] = summary
    state["iteration_count"] = state["iteration_count"] + 1

    return {
        "financial_summary": summary,
        "iteration_count": state["iteration_count"] + 1
        # state
    }

In [132]:
def detect_risks(state: FinancialAdvisorState):
    summary = state["financial_summary"]
    risks = []

    if summary["savings_rate"] < 0.15:
        risks.append("Low savings rate (<15%)")

    top_categories = summary["top_expense_categories"]
    
    if "Dining" in top_categories and top_categories["Dining"] > 0.2 * summary["total_expenses"]:
        risks.append("High dining expenses")

    if summary["net_savings"] < 0:
        risks.append("Spending exceeds income")

    # Simple emergency fund rule
    if summary["net_savings"] < summary["total_expenses"] * 3:
        risks.append("Emergency fund likely insufficient")

    state["risk_flags"] = risks,
    state["iteration_count"] = state["iteration_count"] + 1

    return {
        "risk_flags": risks,
        "iteration_count": state["iteration_count"] + 1
    }

In [133]:
def simulate_scenarios(state: FinancialAdvisorState):
    summary = state["financial_summary"]
    simulations = {}

    income = summary["total_income"]
    expenses = summary["total_expenses"]

    # Scenario 1: Save 10% more
    new_savings = summary["net_savings"] + 0.1 * income
    simulations["increase_savings_10_percent"] = {
        "new_savings": new_savings,
        "new_savings_rate": round(new_savings / income, 2)
    }

    # Scenario 2: Reduce top expense by 15%
    top_category = next(iter(summary["top_expense_categories"]))
    reduction = 0.15 * summary["top_expense_categories"][top_category]

    new_expense = expenses - reduction
    new_savings = income - new_expense

    simulations["reduce_top_category_15_percent"] = {
        "category": top_category,
        "new_savings": new_savings,
        "new_savings_rate": round(new_savings / income, 2)
    }

    state["simulation_results"] = simulations
    state["iteration_count"] = state["iteration_count"] + 1

    return {
        "simulation_results": simulations,
        "iteration_count": state["iteration_count"] + 1
    }

In [135]:
from openai import AzureOpenAI

client = AzureOpenAI(
api_key = PER_API_KEY, # Put your DIAL API Key here
api_version = "2024-02-01",
azure_endpoint = "https://ai-proxy.lab.epam.com"
)

deployment_name = "gpt-4o-mini-2024-07-18"

print(client.chat.completions.create(
model=deployment_name,
temperature=0,
messages=[
{
"role": "user",
"content": "how are you?",
},
],
))

ChatCompletion(id='chatcmpl-DGKMmzshosA9eRvtGOqoIAaSzli20', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I'm just a computer program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_result={'error': {'code': 'content_filter_error', 'message': 'The contents are not filtered'}}, content_filter_results={})], created=1772783908, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier=None, system_fingerprint='fp_eb37e061ec', usage=CompletionUsage(completion_tokens=29, prompt_tokens=11, total_tokens=40, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'prompt_index': 0, '

In [136]:
from openai import AzureOpenAI

client = AzureOpenAI(
api_key = PER_API_KEY, # Put your DIAL API Key here
api_version = "2024-02-01",
azure_endpoint = "https://ai-proxy.lab.epam.com"
)

deployment_name = "gpt-4o-mini-2024-07-18"

In [ ]:
def generate_advice(state: FinancialAdvisorState):

    prompt = f"""
    User Query: {state['user_query']}

    Financial Summary:
    {state['financial_summary']}

    Risk Flags:
    {state['risk_flags']}

    Simulation Results:
    {state['simulation_results']}

    Provide clear, personalized financial advice.
    Be specific and practical.
    """

    response = client.chat.completions.create(
    model=deployment_name,
    temperature=0,
    messages=[
    {
    "role": "user",
    "content": prompt,
    },
    ],
    )

    advice = response.choices[0].message.content

    state["personalized_advice"] = advice,
    state["iteration_count"] = state["iteration_count"] + 1

    return {
        "personalized_advice": advice,
        "iteration_count": state["iteration_count"] + 1
    }

In [142]:
def reflect_and_score(state: FinancialAdvisorState):
    advice = state["personalized_advice"]

    if advice and len(advice) > 100:
        confidence = 0.9
    else:
        confidence = 0.5

    state["confidence_score"] = confidence,
    state["iteration_count"] = state["iteration_count"] + 1

    return {
        "confidence_score": confidence,
        "iteration_count": state["iteration_count"] + 1
    }

In [143]:
MAX_ITERATIONS = 10
CONFIDENCE_THRESHOLD = 0.8

def planner_node(state: FinancialAdvisorState):

    # Safety guard
    if state["iteration_count"] >= MAX_ITERATIONS:
        return {"next_step": "END"}

    # Step 1: Ensure financial analysis exists
    if state.get("financial_summary") is None:
        return {"next_step": "analyze_finances"}

    # Step 2: Ensure risk detection exists
    if state.get("risk_flags") is None:
        return {"next_step": "detect_risks"}

    # Step 3: If user query mentions simulation
    if "save" in state["user_query"].lower() or "what if" in state["user_query"].lower():
        if state.get("simulation_results") is None:
            return {"next_step": "simulate_scenarios"}

    # Step 4: Generate advice if not generated
    if state.get("personalized_advice") is None:
        return {"next_step": "generate_advice"}

    # Step 5: Reflect and score
    if state.get("confidence_score") is None:
        return {"next_step": "reflect_and_score"}

    # Step 6: If low confidence → regenerate advice
    if state["confidence_score"] < 0.8:
        return {"next_step": "generate_advice"}

    # Step 7: All done
    return {"next_step": "END"}

In [144]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(FinancialAdvisorState)

workflow.add_node("planner", planner_node)
workflow.add_node("analyze_finances", analyze_finances)
workflow.add_node("detect_risks", detect_risks)
workflow.add_node("simulate_scenarios", simulate_scenarios)
workflow.add_node("generate_advice", generate_advice)
workflow.add_node("reflect_and_score", reflect_and_score)

workflow.set_entry_point("planner")

In [145]:
workflow.add_conditional_edges(
    "planner",
    lambda state: state["next_step"],
    {
        "analyze_finances": "analyze_finances",
        "detect_risks": "detect_risks",
        "simulate_scenarios": "simulate_scenarios",
        "generate_advice": "generate_advice",
        "reflect_and_score": "reflect_and_score",
        "END": END
    }
)

In [146]:
workflow.add_edge("analyze_finances", "planner")
workflow.add_edge("detect_risks", "planner")
workflow.add_edge("simulate_scenarios", "planner")
workflow.add_edge("generate_advice", "planner")
workflow.add_edge("reflect_and_score", "planner")

In [147]:
app = workflow.compile()

In [149]:
initial_state = {
    "data_path": 'C://Users//Suchi_Kumari//GenAI_Capston//virtual-financial-advisor//data//virtual_financial_advisor_data.csv',
    "user_id": "user_19",
    "user_query": "How can I improve my savings?",
    
    "financial_summary": None,
    "risk_flags": None,
    "simulation_results": None,
    "financial_goal": None,

    "personalized_advice": None,
    "confidence_score": None,

    "iteration_count": 0,
    "next_step": None
}

In [150]:
final_state = app.invoke(initial_state)

print("Advice:")
print(final_state["personalized_advice"])

print("Confidence:")
print(final_state["confidence_score"])

Advice:
Improving your savings can be achieved through a combination of increasing your income, reducing expenses, and optimizing your savings strategy. Here are some specific and practical steps you can take based on your financial summary:

### 1. **Assess Your Emergency Fund**
   - **Risk Flag**: Your emergency fund is likely insufficient. Aim to save at least 3-6 months' worth of living expenses. Given your total expenses of approximately $54,855, this means you should have between $16,456 and $32,913 set aside.
   - **Action**: Prioritize building your emergency fund to at least $20,000 if you haven't already. This will provide a safety net for unexpected expenses.

### 2. **Review and Reduce Expenses**
   - **Top Expense Categories**: Your largest expenses are in groceries, rent, dining, and utilities. Here are some strategies to reduce these costs:
     - **Groceries**: 
       - Create a meal plan and shopping list to avoid impulse purchases.
       - Use coupons and consider b

In [151]:
# analyze_finances(initial_state)

In [152]:
# print(initial_state)

In [153]:
# detect_risks(initial_state)

In [154]:
# simulate_scenarios(initial_state)

In [155]:
# print(initial_state)